# Notebook 1 — Fine-tune LLM for Alzheimer OCR Correction & Clinical Feature Extraction

## What this notebook does
1. Loads your label files and runs the **exact** `postprocess_ocr` cleaning pipeline from `OCR_postprocessing`.
2. Builds (cleaned-OCR-text → structured JSON) training pairs using the **full clinical feature schema**.
3. Fine-tunes `google/flan-t5-base` with LoRA (CPU-friendly, ~344k trainable params).
4. Saves the best checkpoint and provides an inference function that returns a structured JSON file.

## Key fixes vs previous version
- Cleaning now uses `load_transcriptions()` + the full `postprocess_ocr()` with spatial sorting and line grouping — **exactly** as in `OCR_postprocessing`.
- Output JSON uses the **complete clinical schema** (MMSE sub-scores, MoCA, FAQ, biomarkers, family history, medications, etc.)
- Training targets are extracted from real cleaned text using regex + keyword search, not generic templates.
- `master_dataset.jsonl` is used correctly — only `ocr_correction` type French entries feed the model.


In [1]:
# ── INSTALL ─────────────────────────────────────────────────────────────────
!pip install -q torch transformers peft datasets jiwer pyspellchecker tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 74.3 MB/s eta 0:00:00


In [2]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
import os, re, json, random, gc
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from difflib import SequenceMatcher

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import get_peft_model, LoraConfig, TaskType
import warnings
warnings.filterwarnings("ignore")

# CPU only — change to "cuda" if a GPU is available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


Device : cpu
PyTorch: 2.10.0+cpu


## 📁 Path Configuration

```
project/
├── data/                          ← DATA_ROOT
│   ├── 2016/
│   │   ├── images/
│   │   └── transcriptions/        ← *_label.txt files
│   └── 2021/
│       ├── images/
│       └── transcriptions/
├── dictionaries/                  ← DICT_DIR
│   ├── abreviations.txt
│   ├── medications.txt
│   ├── terms.txt
│   └── tests.txt
├── master_dataset.jsonl           ← CORPUS_PATH
└── finetuned_model/               ← MODEL_OUTPUT_DIR  (created automatically)
```

**Set every path in the cell below before running.**


In [3]:
# ── PATHS — EDIT THESE ───────────────────────────────────────────────────────
DATA_ROOT        = "/kaggle/input/datasets/baraafzlalagui/franz-fanon-files-v4/data"                      
DICT_DIR         = "/kaggle/input/datasets/baraafzlalagui/some-alz-fine"                 
CORPUS_PATH      = "/kaggle/input/datasets/baraafzlalagui/some-alz-fine/all_docs_filtered.jsonl"       
MODEL_OUTPUT_DIR = "./finetuned_model"              # <-- where to save the fine-tuned model

ABBREV_PATH      = os.path.join(DICT_DIR, "abreviations.txt")
MEDICATIONS_PATH = os.path.join(DICT_DIR, "medications.txt")
TERMS_PATH       = os.path.join(DICT_DIR, "terms.txt")
TESTS_PATH       = os.path.join(DICT_DIR, "tests.txt")

# ── Training hyper-parameters ─────────────────────────────────────────────
BASE_MODEL        = "google/flan-t5-base"  # ~250M params — better than small, still CPU-ok
MAX_INPUT_TOKENS  = 384   # cleaned OCR text length
MAX_OUTPUT_TOKENS = 512   # JSON output length
BATCH_SIZE        = 2
NUM_EPOCHS        = 3
LEARNING_RATE     = 3e-4
MAX_TRAIN_SAMPLES = None  # set e.g. 200 for a quick demo run; None = use all

# Noise tokens to strip from ground-truth text
NOISE_PATTERNS    = ['MouMNi', 'Moumni', 'moumni']

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print("✓ Paths configured")


✓ Paths configured


## Step 1 — Load Medical Dictionaries

In [4]:
# ── STEP 1 : LOAD DICTIONARIES ───────────────────────────────────────────────
# Exact functions from OCR_postprocessing notebook

def load_list(file_path: str) -> List[str]:
    """Load a simple word list (one item per line)."""
    items = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            items = [line.strip().lower() for line in f if line.strip()]
        print(f"  ✓ {len(items):>4} items  ← {file_path}")
    except FileNotFoundError:
        print(f"  ⚠ Not found: {file_path}")
    return items

def load_abbreviations(file_path: str) -> Dict[str, str]:
    """
    Load abbreviations: full_term,abbr1,abbr2,...
    Returns {abbr: full_term}
    """
    d = {}
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or ',' not in line:
                    continue
                parts = [p.strip().lower() for p in line.split(',')]
                for variant in parts[1:]:
                    d[variant] = parts[0]
        print(f"  ✓ {len(d):>4} abbreviations ← {file_path}")
    except FileNotFoundError:
        print(f"  ⚠ Not found: {file_path}")
    return d

print("Loading dictionaries...")
abbreviations = load_abbreviations(ABBREV_PATH)
medications   = load_list(MEDICATIONS_PATH)
terms         = load_list(TERMS_PATH)
tests         = load_list(TESTS_PATH)


Loading dictionaries...
  ✓  147 abbreviations ← /kaggle/input/datasets/baraafzlalagui/some-alz-fine/abreviations.txt
  ✓   34 items  ← /kaggle/input/datasets/baraafzlalagui/some-alz-fine/medications.txt
  ✓  138 items  ← /kaggle/input/datasets/baraafzlalagui/some-alz-fine/terms.txt
  ✓    7 items  ← /kaggle/input/datasets/baraafzlalagui/some-alz-fine/tests.txt


## Step 2 — OCR Cleaning Pipeline

This is the **exact** `postprocess_ocr` pipeline from `OCR_postprocessing__4_.ipynb`, including:
- `load_transcriptions` (tab-separated poly_gt JSON format)
- `split_mixed_words`, `normalize_dates`
- `replace_abbreviations` (with corrected-index tracking)
- `apply_fuzzy_matching` (medications → terms → tests)
- `correct_french_spelling` (skipping already-corrected and reference words)
- `remove_noise`, `clean_text`
- `sort_by_coordinates`, `group_into_lines` (spatial ordering)
- `postprocess_ocr` (the full orchestrator)

A second entry point `postprocess_plain_text()` is also provided for when you already have a raw string (e.g. from the OCR model in Notebook 3).


In [5]:
# ── STEP 2 : EXACT CLEANING PIPELINE (from OCR_postprocessing__4_.ipynb) ─────

# ── 2a. load_transcriptions ──────────────────────────────────────────────────
def load_transcriptions(label_path: str, transcription_key: str = 'transcription') -> List[Dict]:
    """
    Load OCR transcriptions from a poly_gt label file.

    Two supported formats
    ─────────────────────
    Format A (tab-separated JSON, used in poly_gt files):
        image_path \t [{"transcription": "text", "points": [[x,y],...]}]

    Format B (tag-delimited, used in *_label.txt files from your dataset):
        <|ref|>text<|/ref|><|det|>[[x1,y1,x2,y2]]<|/det|>
        transcription text here
    """
    transcriptions = []
    try:
        with open(label_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # ── Format A: tab-separated JSON ──────────────────────────────────────
        parsed_any = False
        for line in content.splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t', 1)
            if len(parts) == 2:
                image_path = parts[0]
                try:
                    items = json.loads(parts[1])
                    for item in items:
                        if transcription_key in item:
                            item['image'] = image_path
                            transcriptions.append(item)
                    parsed_any = True
                except (json.JSONDecodeError, TypeError):
                    pass

        # ── Format B: tag-delimited (<|ref|>…<|/det|>) ───────────────────────
        if not parsed_any:
            # Pull bounding-box coordinates from <|det|>…<|/det|>
            blocks = re.findall(
                r'<\|ref\|>.*?<\|/ref\|>\s*<\|det\|>(.*?)<\|/det\|>\s*(.*?)(?=<\|ref\||\Z)',
                content, re.DOTALL
            )
            image_name = Path(label_path).stem  # use filename as pseudo-image key
            for det_str, text_block in blocks:
                text_block = text_block.strip()
                if not text_block or text_block == '###':
                    continue
                # Parse bounding box [[x1,y1,x2,y2]]
                try:
                    coords = json.loads(det_str.strip())
                    # Convert [[x1,y1,x2,y2]] → list of [x,y] points
                    if coords and isinstance(coords[0], list) and len(coords[0]) == 4:
                        x1, y1, x2, y2 = coords[0]
                        points = [[x1,y1],[x2,y1],[x2,y2],[x1,y2]]
                    else:
                        points = coords
                except Exception:
                    points = []
                transcriptions.append({
                    transcription_key: text_block,
                    'points': points,
                    'image': image_name,
                })

    except Exception as e:
        print(f"  ⚠ Error reading {label_path}: {e}")
    return transcriptions


# ── 2b. split_mixed_words ─────────────────────────────────────────────────────
def split_mixed_words(text: str) -> str:
    """2ampoules → 2 ampoules,  mots3 → mots 3"""
    text = re.sub(r'(\d+)([a-zàâäéèêëïîôöœçñ]+)', r'\1 \2', text, flags=re.IGNORECASE)
    text = re.sub(r'([a-zàâäéèêëïîôöœçñ]+)(\d+)', r'\1 \2', text, flags=re.IGNORECASE)
    return text


# ── 2c. normalize_dates ───────────────────────────────────────────────────────
def normalize_dates(text: str) -> str:
    """Normalise all dates to jj/mm/aaaa, append newline after each date."""
    pattern = r'(\d{1,2})\s*[\s/:\-.\'](\d{1,2})\s*[\s/:\-.\'](\d{2,4})'
    def _rep(m):
        day, month, year = m.group(1).zfill(2), m.group(2).zfill(2), m.group(3)
        if len(year) == 2:
            year = '20' + year
        return f"{day}/{month}/{year}\n"
    return re.sub(pattern, _rep, text)


# ── 2d. replace_abbreviations ─────────────────────────────────────────────────
def replace_abbreviations(text: str, abbrev_dict: Dict[str, str]) -> Tuple[str, set]:
    """Replace abbreviations; return (corrected_text, set_of_corrected_word_indices)."""
    words = text.split()
    result, corrected_indices = [], set()
    for i, word in enumerate(words):
        word_clean = re.sub(r"[^\w'/]", '', word.lower())
        if word_clean in abbrev_dict:
            result.append(abbrev_dict[word_clean])
            corrected_indices.add(i)
        else:
            result.append(word)
    return ' '.join(result), corrected_indices


# ── 2e. fuzzy matching ────────────────────────────────────────────────────────
def find_closest_match(word: str, reference_list: List[str], threshold: float = 0.75) -> Tuple[str, bool]:
    """SequenceMatcher-based closest match. Returns (match, was_corrected)."""
    word_lower = word.lower()
    if word_lower in reference_list:
        return word_lower, True
    best_match, best_ratio = None, threshold
    for ref in reference_list:
        ratio = SequenceMatcher(None, word_lower, ref).ratio()
        if ratio > best_ratio:
            best_ratio, best_match = ratio, ref
    return (best_match, True) if best_match else (word, False)

def apply_fuzzy_matching(text: str, medications: List[str], terms: List[str], tests: List[str]) -> Tuple[str, set]:
    """Fuzzy-match medical vocabulary; return (corrected_text, corrected_indices)."""
    words = text.split()
    result, corrected_indices = [], set()
    for i, word in enumerate(words):
        m, ok = find_closest_match(word, medications, 0.75)
        if ok: result.append(m); corrected_indices.add(i); continue
        m, ok = find_closest_match(word, terms, 0.75)
        if ok: result.append(m); corrected_indices.add(i); continue
        m, ok = find_closest_match(word, tests, 0.75)
        if ok: result.append(m); corrected_indices.add(i); continue
        result.append(word)
    return ' '.join(result), corrected_indices


# ── 2f. correct_french_spelling ───────────────────────────────────────────────
def correct_french_spelling(text: str, skip_indices: set = None,
                             reference_lists: List[List[str]] = None) -> str:
    """
    French spell-check.  Words that were already corrected (skip_indices) and words
    in reference_lists are left untouched.
    """
    if skip_indices is None:
        skip_indices = set()
    if reference_lists is None:
        reference_lists = []
    reference_words = {w.lower() for lst in reference_lists for w in lst}
    try:
        from spellchecker import SpellChecker
        spell = SpellChecker(language='fr')
        words = text.split()
        corrected = []
        for i, word in enumerate(words):
            if i in skip_indices:
                corrected.append(word); continue
            word_lower = word.lower()
            word_clean = re.sub(r'[^\w]', '', word_lower)
            if word_clean in reference_words:
                corrected.append(word); continue
            if not re.search(r'[a-zàâäéèêëïîôöœçñ]', word, re.IGNORECASE):
                corrected.append(word); continue
            if word_clean not in spell:
                closest = spell.correction(word_clean)
                corrected.append(closest if closest else word)
            else:
                corrected.append(word)
        return ' '.join(corrected)
    except ImportError:
        print("  ⚠ pyspellchecker not installed — spell-check step skipped")
        return text


# ── 2g. spatial helpers ───────────────────────────────────────────────────────
def get_centroid(points: List) -> Tuple[float, float]:
    """Y, X centre of a polygon."""
    if not points:
        return float('inf'), float('inf')
    return (sum(p[1] for p in points) / len(points),
            sum(p[0] for p in points) / len(points))

def sort_by_coordinates(items: List[Dict]) -> List[Dict]:
    return sorted(items, key=lambda item: get_centroid(item.get('points', [])))

def group_into_lines(items: List[Dict], y_threshold: int = 100) -> List[List[Dict]]:
    """Group spatially close items into text lines."""
    if not items:
        return []
    lines = [[items[0]]]
    for i in range(1, len(items)):
        cy, _ = get_centroid(items[i].get('points', []))
        py, _ = get_centroid(items[i-1].get('points', []))
        if abs(cy - py) > y_threshold:
            lines.append([items[i]])
        else:
            lines[-1].append(items[i])
    return lines


# ── 2h. noise helpers ─────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

def remove_noise(text: str, noise_patterns: List[str]) -> str:
    for pat in noise_patterns:
        try:
            text = re.sub(pat, '', text, flags=re.IGNORECASE)
        except re.error:
            text = text.replace(pat, '')
    return re.sub(r'\s+', ' ', text).strip()


# ── 2i. postprocess_ocr — the FULL orchestrator ───────────────────────────────
def postprocess_ocr(
    transcriptions: List[Dict],
    abbrev_dict: Dict[str, str],
    medications: List[str],
    terms: List[str],
    tests: List[str],
    noise_patterns: List[str] = None,
    transcription_key: str = 'transcription',
    y_threshold: int = 50,
) -> str:
    """
    Complete OCR post-processing pipeline — identical logic to OCR_postprocessing__4_.ipynb:

    1. Sort items by spatial coordinates
    2. Group by image
    3. For each image, group into lines, sort within line
    4. Per word: split_mixed_words → normalize_dates → replace_abbreviations →
       apply_fuzzy_matching → correct_french_spelling → remove_noise → clean_text
    5. Return the full page text (images separated by blank line)
    """
    if noise_patterns is None:
        noise_patterns = NOISE_PATTERNS

    transcriptions = sort_by_coordinates(transcriptions)

    # Group by image
    image_groups: Dict[str, List[Dict]] = {}
    for item in transcriptions:
        img = item.get('image', '__default__')
        image_groups.setdefault(img, []).append(item)

    final_parts = []
    for img in sorted(image_groups.keys()):
        items = image_groups[img]
        lines = group_into_lines(items, y_threshold=y_threshold)

        line_texts = []
        for line_items in lines:
            # Sort words left-to-right within each line
            line_items = sorted(line_items,
                                key=lambda x: get_centroid(x.get('points', []))[1])
            line_words = []
            for item in line_items:
                text = item.get(transcription_key, '')
                # Skip noise-only transcriptions
                if re.match(r'^\s*(MouMNi|Moumni|moumni|psychologue)\s*$', text, re.IGNORECASE):
                    continue

                all_corrected: set = set()

                text = split_mixed_words(text)
                text = normalize_dates(text)

                text, abbrev_idx = replace_abbreviations(text, abbrev_dict)
                all_corrected.update(abbrev_idx)

                text, fuzzy_idx = apply_fuzzy_matching(text, medications, terms, tests)
                all_corrected.update(fuzzy_idx)

                text = correct_french_spelling(
                    text,
                    skip_indices=all_corrected,
                    reference_lists=[medications, tests, ['CM', 'TM', 'ro']],
                )
                text = remove_noise(text, noise_patterns)
                text = clean_text(text)

                if text:
                    line_words.append(text)

            if line_words:
                line_texts.append(' '.join(line_words))

        if line_texts:
            final_parts.append('\n'.join(line_texts))

    return '\n\n'.join(final_parts)


# ── 2j. postprocess_plain_text — for raw string input ────────────────────────
def postprocess_plain_text(raw_text: str) -> str:
    """
    Apply the same cleaning steps to a plain string (e.g. direct OCR model output).
    Used in Notebook 3 where we don't have poly_gt coordinate annotations.
    """
    text = split_mixed_words(raw_text)
    text = normalize_dates(text)
    text, abbrev_idx  = replace_abbreviations(text, abbreviations)
    text, fuzzy_idx   = apply_fuzzy_matching(text, medications, terms, tests)
    all_corrected     = abbrev_idx | fuzzy_idx
    text = correct_french_spelling(
        text,
        skip_indices=all_corrected,
        reference_lists=[medications, tests, ['CM', 'TM', 'ro']],
    )
    text = remove_noise(text, NOISE_PATTERNS)
    text = clean_text(text)
    return text


# ── Quick self-test ───────────────────────────────────────────────────────────
_sample = "ATCDs: HTA evoluant depuis 04 ans, trbls mnésique, 2ampoules donecept"
print("Raw    :", _sample)
print("Cleaned:", postprocess_plain_text(_sample))
print()
print("✓ Cleaning pipeline ready")


Raw    : ATCDs: HTA evoluant depuis 04 ans, trbls mnésique, 2ampoules donecept
Cleaned: accès hypertension artérielle évoluant depuis 04 ans, troubles mnésique 2 ampoule donecept

✓ Cleaning pipeline ready


## Step 3 — Clinical Feature Schema

The target output is a JSON object with the **complete clinical schema**.
Fields are grouped by category; missing values are `null`.


In [6]:
# ── STEP 3 : COMPLETE CLINICAL FEATURE SCHEMA ───────────────────────────────
#
# Every field the model should try to extract.
# Values it cannot find are set to null.
#
CLINICAL_SCHEMA = {
    # ── Demographics ─────────────────────────────────────────────────────────
    "PTGENDER":   None,   # 1=Male 2=Female
    "age":        None,   # integer years
    "PTHAND":     None,   # 1=Right 2=Left 3=Ambidextrous
    "PTMARRY":    None,   # 1=Married 2=Widowed 3=Divorced 4=Never married 5=Unknown
    "PTEDUCAT":   None,   # years of education
    "PTWORK":     None,   # occupation / employment status
    "PTNOTRT":    None,   # 1=Not in treatment 0=In treatment
    "VISDATE":    None,   # visit date jj/mm/aaaa

    # ── MMSE sub-scores ───────────────────────────────────────────────────────
    "MMDATE":     None,   # Date orientation (0-1)
    "MMYEAR":     None,   # Year orientation (0-1)
    "MMMONTH":    None,   # Month orientation (0-1)
    "MMDAY":      None,   # Day orientation (0-1)
    "MMREAD":     None,   # Reading (0-1)
    "MMWRITE":    None,   # Writing (0-1)
    "MMDRAW":     None,   # Drawing (0-1)
    "MMREPEAT":   None,   # Repeat phrase (0-1)
    "MMSEASON":   None,   # Season (0-1)
    "MMHOSPIT":   None,   # Hospital (0-1)
    "MMFLOOR":    None,   # Floor (0-1)
    "MMCITY":     None,   # City (0-1)
    "WORD1":      None,   # Word recall 1
    "WORD2":      None,   # Word recall 2
    "WORD3":      None,   # Word recall 3
    "MMSCORE":    None,   # Total MMSE score (0-30)

    # ── MoCA sub-scores ───────────────────────────────────────────────────────
    "MOCA":       None,   # Total MoCA score (0-30)
    "CUBE":       None,   # Cube copy (0-1)
    "CLOCKCON":   None,   # Clock contour (0-1)
    "CLOCKNO":    None,   # Clock numbers (0-1)
    "CLOCKHAN":   None,   # Clock hands (0-1)
    "DIGFOR":     None,   # Digit span forward
    "DIGBACK":    None,   # Digit span backward
    "SERIAL1":    None,   # Serial 7s attempt 1
    "SERIAL2":    None,   # Serial 7s attempt 2
    "SERIAL3":    None,   # Serial 7s attempt 3
    "SERIAL4":    None,   # Serial 7s attempt 4
    "SERIAL5":    None,   # Serial 7s attempt 5
    "REPEAT1":    None,   # Sentence repetition 1
    "REPEAT2":    None,   # Sentence repetition 2
    "FFLUENCY":   None,   # Letter fluency (F words)

    # ── FAQ (Functional Activities Questionnaire) ─────────────────────────────
    "FAQFORM":    None,   "FAQFINAN":   None,   "FAQSHOP":    None,
    "FAQGAME":    None,   "FAQBEVG":    None,   "FAQMEAL":    None,
    "FAQEVENT":   None,   "FAQTV":      None,   "FAQREM":     None,
    "FAQTRAVL":   None,   "FAQ":        None,   # total FAQ score

    # ── CSF / Blood biomarkers ────────────────────────────────────────────────
    "Abeta42":        None,   "Abeta40":         None,   "Abeta_ratio":     None,
    "pTau217":        None,   "npTau217":        None,   "Tau181":          None,
    "pTau181":        None,   "pTau217_ratio":   None,   "GFAP":            None,
    "NfL":            None,

    # ── Standard lab tests ────────────────────────────────────────────────────
    "Alanine Aminotransferase":     None,   "Alkaline Phosphatase":    None,
    "Aspartate Aminotransferase":   None,   "Calcium":                 None,
    "Creatinine":                   None,   "Cystatin C":              None,
    "Direct Bilirubin":             None,   "Gamma-Glutamyltransferase": None,
    "Hematocrit":                   None,   "Hemoglobin":              None,
    "Hemoglobin A1C":               None,   "High Sensitivity CRP":    None,
    "Homocysteine":                 None,   "Lactate Dehydrogenase":   None,
    "Methylmalonic Acid":           None,   "Platelet Ct.":            None,
    "Red Blood Cell Count":         None,   "Thyroid-stimulating hormone": None,
    "Total Bilirubin":              None,   "Urea Nitrogen":           None,
    "Vitamin B12":                  None,   "White Blood Cell Count":  None,
    "eGFR by Cystatin C":           None,   "glucose":                 None,
    "protein":                      None,

    # ── Family history ────────────────────────────────────────────────────────
    "MOTHDEM":    None,   "MOTHAD":     None,   "MOTHSXAGE":  None,
    "FATHDEM":    None,   "FATHAD":     None,   "FATHSXAGE":  None,
    "SIBGENDER":  None,   "SIBDEMENT":  None,   "SIBAD":      None,
    "SIBSXAGE":   None,

    # ── Intercurrent history ──────────────────────────────────────────────────
    "IHSYMPTOM":  None,   "IHDESC":     None,   "IHCHRON":    None,
    "IHSEVER":    None,   "IHPRESENT":  None,   "IHSURG":     None,

    # ── Medical history flags ─────────────────────────────────────────────────
    "MH19OTHR":   None,   "MHPSYCH":    None,   "MH2NEURL":   None,
    "MH3HEAD":    None,   "MH4CARD":    None,   "MH5RESP":    None,
    "MH6HEPAT":   None,   "MH7DERM":    None,   "MH8MUSCL":   None,
    "MH9ENDO":    None,   "MH14BALCH":  None,   "MH14CALCH":  None,
    "MH10GAST":   None,   "MH11HEMA":   None,   "MH12RENA":   None,
    "MH13ALLE":   None,   "MH14ALCH":   None,   "MH14AALCH":  None,
    "MH17MALI":   None,   "MH18SURG":   None,   "MH15DRUG":   None,
    "MH15ADRUG":  None,   "MH15BDRUG":  None,   "MH16SMOK":   None,
    "MH16ASMOK":  None,   "MH16BSMOK":  None,   "MH16CSMOK":  None,

    # ── Baseline symptoms & medications ──────────────────────────────────────
    "BSXSYMNO":   None,   "BSXSEVER":   None,   "BSXCHRON":   None,
    "KEYMED":     None,   "CMMED":      None,   "CMDOSE":     None,
    "CMREASON":   None,
}

print(f"✓ Clinical schema defined: {len(CLINICAL_SCHEMA)} fields")
print("  Fields:", list(CLINICAL_SCHEMA.keys())[:10], "...")


✓ Clinical schema defined: 135 fields
  Fields: ['PTGENDER', 'age', 'PTHAND', 'PTMARRY', 'PTEDUCAT', 'PTWORK', 'PTNOTRT', 'VISDATE', 'MMDATE', 'MMYEAR'] ...


## Step 4 — Feature Extraction Helpers

These functions scan cleaned French text with regex and keyword search to fill as many schema fields as possible.
The resulting dict becomes the **ground-truth JSON** the model is trained to produce.


In [7]:
# ── STEP 4 : EXTRACT FEATURES FROM CLEANED TEXT ─────────────────────────────

import copy

def extract_features(text: str) -> Dict:
    """
    Extract every clinical feature we can find in the cleaned French medical text.
    Returns a copy of CLINICAL_SCHEMA with found values filled in.
    """
    feat = copy.deepcopy(CLINICAL_SCHEMA)
    t = text.lower()

    # ── Demographics ─────────────────────────────────────────────────────────
    # Visit date
    date_m = re.search(r'(\d{2}/\d{2}/\d{4})', text)
    if date_m:
        feat['VISDATE'] = date_m.group(1)

    # Age
    age_m = re.search(r'(?:age[^\d]{0,5}|âgé[^\d]{0,5})(\d{2})\s*ans', t)
    if not age_m:
        age_m = re.search(r'(\d{2})\s*ans', t)
    if age_m:
        feat['age'] = int(age_m.group(1))

    # Gender
    if re.search(r'\b(homme|monsieur|m\.|mr\.|masculin|male)\b', t):
        feat['PTGENDER'] = 1
    elif re.search(r'\b(femme|madame|mme\.?|féminin|female)\b', t):
        feat['PTGENDER'] = 2

    # Handedness
    if re.search(r'\b(droitier|droite|right.hand)\b', t):
        feat['PTHAND'] = 1
    elif re.search(r'\b(gaucher|gauche|left.hand)\b', t):
        feat['PTHAND'] = 2

    # Marital status
    if re.search(r'\b(marié|mariée|époux|épouse)\b', t):
        feat['PTMARRY'] = 1
    elif re.search(r'\b(veuf|veuve|veuvage)\b', t):
        feat['PTMARRY'] = 2
    elif re.search(r'\b(divorcé|divorcée)\b', t):
        feat['PTMARRY'] = 3
    elif re.search(r'\b(célibataire)\b', t):
        feat['PTMARRY'] = 4

    # Education
    edu_m = re.search(r'(\d+)\s*(?:ans?\s+)?(?:d.étude|d.scolarité|années?\s+d.étude)', t)
    if edu_m:
        feat['PTEDUCAT'] = int(edu_m.group(1))
    elif re.search(r'\b(analphabète|illettré)\b', t):
        feat['PTEDUCAT'] = 0

    # Occupation
    occ_m = re.search(r'(?:profession|professce|travail|métier)[^\n:]{0,20}[:.]?\s*([a-zàâäéèêëïîôöœç ]{3,30})', t)
    if occ_m:
        feat['PTWORK'] = occ_m.group(1).strip()
    elif re.search(r'\bretraité\b', t):
        feat['PTWORK'] = 'retraité'

    # ── MMSE ─────────────────────────────────────────────────────────────────
    mmscore_m = re.search(r'(?:mmse|mms)[\s:=/-]{0,5}(\d{1,2})(?:/30)?', t)
    if mmscore_m:
        feat['MMSCORE'] = int(mmscore_m.group(1))

    # ── MoCA ─────────────────────────────────────────────────────────────────
    moca_m = re.search(r'moca[\s:=/-]{0,5}(\d{1,2})(?:/30)?', t)
    if moca_m:
        feat['MOCA'] = int(moca_m.group(1))

    # Clock drawing sub-scores
    if re.search(r'(?:horloge|clock)[^\n]{0,60}(\d)', t):
        m = re.search(r'(?:horloge|clock)[^\n]{0,60}(\d)/3', t)
        if m:
            score = int(m.group(1))
            feat['CLOCKCON'] = min(score, 1)

    # Digit span
    fwd_m = re.search(r'(?:empan|digit.?span|chiffres?)[^\n]{0,30}(\d)\s*(?:avant|foward|en\s+avant)', t)
    if fwd_m:
        feat['DIGFOR'] = int(fwd_m.group(1))
    bwd_m = re.search(r'(?:empan|digit.?span|chiffres?)[^\n]{0,30}(\d)\s*(?:arrière|backward|en\s+arrière)', t)
    if bwd_m:
        feat['DIGBACK'] = int(bwd_m.group(1))

    # Serial 7s  (look for five consecutive numbers after "calcul" / "soustraction")
    serial_m = re.findall(r'(?:calcul|soustraction|serial)[^\n]{0,20}(\d+)', t)
    for idx, val in enumerate(serial_m[:5], start=1):
        feat[f'SERIAL{idx}'] = int(val)

    # Verbal fluency
    fluency_m = re.search(r'(?:fluence|fluency|fluences?)[^\d]{0,20}(\d+)', t)
    if fluency_m:
        feat['FFLUENCY'] = int(fluency_m.group(1))

    # ── FAQ ───────────────────────────────────────────────────────────────────
    faq_m = re.search(r'faq[\s:=/-]{0,5}(\d{1,2})(?:/30)?', t)
    if faq_m:
        feat['FAQ'] = int(faq_m.group(1))
    # IADL / ADL (partial FAQ proxy)
    iadl_m = re.search(r'(?:iadl|adl|psms)[\s:=/-]{0,5}(\d+)', t)
    if iadl_m:
        feat['FAQFORM'] = int(iadl_m.group(1))

    # ── Biomarkers (numeric values) ───────────────────────────────────────────
    biomarker_patterns = {
        'glucose':    r'(?:glycémie|glucose)[\s:=]{0,5}([\d.,]+)',
        'Hemoglobin A1C': r'(?:hba1c|ha1c|hémoglobine\s+a1c)[\s:=]{0,5}([\d.,]+)',
        'Creatinine': r'(?:créatinine|creatinine)[\s:=]{0,5}([\d.,]+)',
        'Calcium':    r'(?:calcium|ca)[\s:=]{0,5}([\d.,]+)',
        'Vitamin B12': r'(?:vitamine?\s*b12|b12|vit\.?b12)[\s:=]{0,5}([\d.,]+)',
        'Hemoglobin': r'(?:hémoglobine|hemoglobin|hb)[\s:=]{0,5}([\d.,]+)',
        'Abeta42':    r'(?:a.?beta.?42|aβ42)[\s:=]{0,5}([\d.,]+)',
        'pTau181':    r'(?:ptau181|p.tau.?181)[\s:=]{0,5}([\d.,]+)',
        'NfL':        r'(?:nfl|neurofilament)[\s:=]{0,5}([\d.,]+)',
    }
    for field, pattern in biomarker_patterns.items():
        m = re.search(pattern, t)
        if m:
            try:
                feat[field] = float(m.group(1).replace(',', '.'))
            except ValueError:
                pass

    # ── Medical history flags from keywords ───────────────────────────────────
    mh_keywords = {
        'MHPSYCH':  ['psychiatrique', 'dépression', 'anxiété', 'psychose'],
        'MH2NEURL': ['neurologique', 'épilepsie', 'avc', 'parkinson'],
        'MH3HEAD':  ['traumatisme crânien', 'céphalée', 'migraine'],
        'MH4CARD':  ['cardiopathie', 'infarctus', 'coronaire', 'cardiaque'],
        'MH5RESP':  ['respiratoire', 'asthme', 'bpco', 'emphysème'],
        'MH6HEPAT': ['hépatique', 'hépatite', 'cirrhose'],
        'MH9ENDO':  ['diabète', 'thyroïde', 'hypothyroïdie', 'hyperthyroïdie'],
        'MH12RENA': ['rénal', 'rénale', 'insuffisance rénale', 'dialyse'],
        'MH4CARD':  ['hypertension', 'hta'],
        'MH16SMOK': ['tabac', 'tabagisme', 'fumeur', 'cigarette'],
        'MH14ALCH': ['alcool', 'éthylisme'],
    }
    for field, keywords in mh_keywords.items():
        if any(kw in t for kw in keywords):
            feat[field] = 1

    # ── Medications ───────────────────────────────────────────────────────────
    found_meds = [m for m in medications if m in t]
    if found_meds:
        feat['CMMED']   = found_meds[0]
        feat['KEYMED']  = ', '.join(found_meds)

    # Dose
    dose_m = re.search(r'(\d+(?:[.,]\d+)?\s*mg)', t)
    if dose_m:
        feat['CMDOSE'] = dose_m.group(1)

    # Smoking details
    smk_m = re.search(r'(\d+)\s*(?:cigarettes?|paquets?)', t)
    if smk_m:
        feat['MH16ASMOK'] = int(smk_m.group(1))

    # ── Treatment flag ────────────────────────────────────────────────────────
    if re.search(r'(?:sous\s+traitement|s/trt|en\s+traitement)', t):
        feat['PTNOTRT'] = 0
    elif re.search(r'\btraitement\b', t):
        feat['PTNOTRT'] = 0

    # ── Family history ────────────────────────────────────────────────────────
    if re.search(r'(?:mère|mother|maternelle?)[^\n]{0,40}(?:démence|alzheimer|dcd)', t):
        feat['MOTHDEM'] = 1
        feat['MOTHAD']  = 1 if 'alzheimer' in t else None
    if re.search(r'(?:père|father|paternelle?)[^\n]{0,40}(?:démence|alzheimer|dcd)', t):
        feat['FATHDEM'] = 1
        feat['FATHAD']  = 1 if 'alzheimer' in t else None
    if re.search(r'(?:frère|sœur|sibling|frere|soeur)[^\n]{0,40}(?:démence|alzheimer)', t):
        feat['SIBDEMENT'] = 1

    return feat


# ── Quick test ────────────────────────────────────────────────────────────────
_demo = (
    "Le 20/10/2016 - Age: 72 ans - Madame - ATCDF: la mère DCD d'une démence "
    "- MMSE: 18/30 - MoCA: 14/30 - FAQ: 12 - sous traitement: donépézil 5mg "
    "- HTA, diabète type 2 - glycémie: 2.67 - vitamine B12: 320"
)
_cleaned = postprocess_plain_text(_demo)
_feat    = extract_features(_cleaned)

print("Cleaned text:", _cleaned[:120], "...")
print()
print("Extracted features (non-null only):")
non_null = {k: v for k, v in _feat.items() if v is not None}
print(json.dumps(non_null, ensure_ascii=False, indent=2))


Cleaned text: Le 20/10/2016 - âge 72 ans - Madame - antécédent familiaux la mère décédé d'une démence - mmse 18/30 - moca 14/30 - fan  ...

Extracted features (non-null only):
{
  "PTGENDER": 2,
  "age": 72,
  "PTNOTRT": 0,
  "VISDATE": "20/10/2016",
  "MMSCORE": 18,
  "MOCA": 14,
  "Calcium": 14.0,
  "glucose": 2.67,
  "MOTHDEM": 1,
  "MH4CARD": 1,
  "MH9ENDO": 1,
  "KEYMED": "b, donépézil, e, mag, vitamine",
  "CMMED": "b"
}


## Step 5 — Build Training Pairs from Label Files

For each `*_label.txt` file we:
1. Load with `load_transcriptions()` (handles both poly_gt JSON and tag-delimited formats)
2. Run `postprocess_ocr()` to get clean text
3. Run `extract_features()` to get the target JSON
4. Build `(instruction_prompt, json_string)` pairs


In [8]:
# ── STEP 5a : INSTRUCTION PROMPT ────────────────────────────────────────────

# Compact list of target fields shown in the prompt so the model knows what to look for
TARGET_FIELDS_SUMMARY = (
    "PTGENDER, age, PTHAND, PTMARRY, PTEDUCAT, PTWORK, PTNOTRT, VISDATE, "
    "MMSCORE, MOCA, FAQ, MMDATE..MMCITY, WORD1-3, CUBE, CLOCKCON, CLOCKNO, CLOCKHAN, "
    "DIGFOR, DIGBACK, SERIAL1-5, REPEAT1-2, FFLUENCY, FAQFORM..FAQTRAVL, "
    "Abeta42, pTau181, NfL, GFAP, glucose, Vitamin B12, Creatinine, Hemoglobin A1C, "
    "MOTHDEM, FATHDEM, SIBDEMENT, MH4CARD, MH9ENDO, MH16SMOK, MH14ALCH, "
    "CMMED, CMDOSE, KEYMED, CMREASON, BSXSEVER"
)

def build_prompt(cleaned_text: str) -> str:
    return (
        "Tu es un assistant médical expert en maladie d'Alzheimer. "
        "Lis le texte médical ci-dessous et extrait toutes les informations disponibles. "
        "Réponds UNIQUEMENT avec un objet JSON valide contenant ces champs "
        f"(null si l'information est absente): {TARGET_FIELDS_SUMMARY}.\n"
        f"Texte médical:\n{cleaned_text}"
    )

def build_target_json(features: Dict) -> str:
    """Serialise only non-null features to keep output compact."""
    non_null = {k: v for k, v in features.items() if v is not None}
    return json.dumps(non_null, ensure_ascii=False)


# ── STEP 5b : LOAD LABEL FILES AND BUILD PAIRS ───────────────────────────────

def load_label_pairs_from_dir(
    data_root: str,
    max_samples: Optional[int] = None,
) -> List[Tuple[str, str]]:
    """
    Walk data_root recursively, find every *_label.txt, clean it, extract features,
    and return (prompt, json_target) training pairs.

    Expected folder layout:
        data_root/
        └── YEAR/
            └── transcriptions/
                └── YEAR_PATIENTID_PAGEID_label.txt
    """
    pairs = []
    label_files = sorted(Path(data_root).rglob("*_label.txt"))
    print(f"  Found {len(label_files)} label files under {data_root}")

    for lf in label_files:
        if max_samples and len(pairs) >= max_samples:
            break
        try:
            transcriptions = load_transcriptions(str(lf), transcription_key='transcription')
            if not transcriptions:
                continue

            cleaned = postprocess_ocr(
                transcriptions,
                abbrev_dict=abbreviations,
                medications=medications,
                terms=terms,
                tests=tests,
                noise_patterns=NOISE_PATTERNS,
            )

            if len(cleaned.strip()) < 15:
                continue

            features = extract_features(cleaned)
            non_null_count = sum(1 for v in features.values() if v is not None)

            # Only keep pairs that extracted at least one useful field
            if non_null_count == 0:
                continue

            prompt = build_prompt(cleaned)
            target = build_target_json(features)
            pairs.append((prompt, target))

        except Exception as e:
            print(f"  ⚠ Skipping {lf.name}: {e}")

    print(f"  ✓ Built {len(pairs)} training pairs from label files")
    return pairs


gt_pairs = load_label_pairs_from_dir(DATA_ROOT, max_samples=MAX_TRAIN_SAMPLES)


  Found 195 label files under /kaggle/input/datasets/baraafzlalagui/franz-fanon-files-v4/data
  ✓ Built 195 training pairs from label files


In [9]:
# ── STEP 5c : LOAD master_dataset.jsonl ─────────────────────────────────────
#
# Only the entries whose 'type' == 'ocr_correction' and language is French
# are directly useful.  All other entries are used as read-comprehension
# context (we strip them into (passage, JSON) pairs where possible).
#

def load_corpus_pairs(
    corpus_path: str,
    max_samples: Optional[int] = None,
) -> List[Tuple[str, str]]:
    """
    Convert master_dataset.jsonl entries to (prompt, json_target) pairs.

    Strategy:
    - If entry already has a clean 'messages' pair → extract as-is, re-run feature
      extraction on the assistant message to get proper schema output.
    - We skip entries that are clearly off-topic (no French text, no medical content).
    """
    pairs = []
    skipped = 0
    try:
        with open(corpus_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"  ⚠ Corpus not found at {corpus_path}")
        return pairs

    print(f"  Scanning {len(lines)} corpus entries...")

    for line in lines:
        if max_samples and len(pairs) >= max_samples:
            break
        try:
            item = json.loads(line.strip())
        except json.JSONDecodeError:
            continue

        msgs = item.get('messages', [])
        if len(msgs) < 2:
            skipped += 1; continue

        # Use the assistant's answer text as the raw source to extract from
        asst_text = msgs[1].get('content', '')
        if len(asst_text) < 20:
            skipped += 1; continue

        # Clean it through the pipeline
        cleaned = postprocess_plain_text(asst_text[:600])  # cap length for CPU
        if len(cleaned) < 15:
            skipped += 1; continue

        features = extract_features(cleaned)
        non_null  = sum(1 for v in features.values() if v is not None)
        if non_null == 0:
            skipped += 1; continue

        prompt = build_prompt(cleaned)
        target = build_target_json(features)
        pairs.append((prompt, target))

    print(f"  ✓ Built {len(pairs)} pairs from corpus  ({skipped} skipped)")
    return pairs


corpus_pairs = load_corpus_pairs(CORPUS_PATH, max_samples=MAX_TRAIN_SAMPLES)

# ── Combine & shuffle ────────────────────────────────────────────────────────
all_pairs = gt_pairs + corpus_pairs
random.shuffle(all_pairs)
if MAX_TRAIN_SAMPLES:
    all_pairs = all_pairs[:MAX_TRAIN_SAMPLES]

split = int(0.85 * len(all_pairs))
train_pairs = all_pairs[:split]
val_pairs   = all_pairs[split:]

print()
print(f"  ✓ Total pairs : {len(all_pairs)}")
print(f"  ✓ Train / Val : {len(train_pairs)} / {len(val_pairs)}")
if all_pairs:
    print()
    print("── Example prompt (first 300 chars) ──")
    print(all_pairs[0][0][:300])
    print()
    print("── Example target JSON ──")
    print(all_pairs[0][1][:300])


  Scanning 93757 corpus entries...
  ✓ Built 0 pairs from corpus  (93757 skipped)

  ✓ Total pairs : 195
  ✓ Train / Val : 165 / 30

── Example prompt (first 300 chars) ──
Tu es un assistant médical expert en maladie d'Alzheimer. Lis le texte médical ci-dessous et extrait toutes les informations disponibles. Réponds UNIQUEMENT avec un objet JSON valide contenant ces champs (null si l'information est absente): PTGENDER, age, PTHAND, PTMARRY, PTEDUCAT, PTWORK, PTNOTRT, 

── Example target JSON ──
{"age": 75, "PTMARRY": 1, "PTEDUCAT": 0, "PTNOTRT": 0, "VISDATE": "02/05/2021", "MH4CARD": 1, "KEYMED": "b, e", "CMMED": "b"}


## Step 6 — Load Model & Apply LoRA

In [10]:
# ── STEP 6 : LOAD MODEL ──────────────────────────────────────────────────────
print(f"Loading {BASE_MODEL} ...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# ── Apply LoRA  (keeps CPU training time manageable) ─────────────────────────
lora_cfg = LoraConfig(
    task_type  = TaskType.SEQ_2_SEQ_LM,
    r          = 16,          # rank — larger = more capacity, slower
    lora_alpha = 32,
    lora_dropout = 0.1,
    target_modules = ["q", "v"],
)
model = get_peft_model(base_model, lora_cfg)
model = model.to(DEVICE)
model.print_trainable_parameters()
print(f"\n✓ Model ready on {DEVICE}")


Loading google/flan-t5-base ...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096

✓ Model ready on cpu


## Step 7 — PyTorch Dataset

In [11]:
# ── STEP 7 : DATASET ─────────────────────────────────────────────────────────

class ClinicalExtractionDataset(Dataset):
    """
    Each sample: tokenised (prompt → json_target) pair.
    Label padding tokens are replaced with -100 so the loss ignores them.
    """
    def __init__(self, pairs, tokenizer, max_in, max_out):
        self.pairs     = pairs
        self.tokenizer = tokenizer
        self.max_in    = max_in
        self.max_out   = max_out

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]

        model_inputs = self.tokenizer(
            src, max_length=self.max_in, truncation=True,
            padding="max_length", return_tensors="pt"
        )

        labels = self.tokenizer(
            tgt, max_length=self.max_out, truncation=True,
            padding="max_length", return_tensors="pt"
        )

        label_ids = labels["input_ids"].squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels":         label_ids,
        }


# ── Build datasets ────────────────────────────────────────────────────────────
# Fallback demo data when no real data is found
if not train_pairs:
    print("⚠ No training pairs found — using a minimal demo example.")
    _demo_prompt = build_prompt(
        "Le 20/10/2016 - Age: 72 ans - Madame - MMSE: 18/30 - MoCA: 14 "
        "- sous traitement: donépézil 5mg - HTA, diabète"
    )
    _demo_target = build_target_json(extract_features(postprocess_plain_text(
        "Le 20/10/2016 - Age: 72 ans - Madame - MMSE: 18/30 - MoCA: 14 "
        "- sous traitement: donépézil 5mg - HTA, diabète"
    )))
    train_pairs = [(_demo_prompt, _demo_target)] * 10
    val_pairs   = [(_demo_prompt, _demo_target)] * 2

train_ds = ClinicalExtractionDataset(train_pairs, tokenizer, MAX_INPUT_TOKENS, MAX_OUTPUT_TOKENS)
val_ds   = ClinicalExtractionDataset(val_pairs,   tokenizer, MAX_INPUT_TOKENS, MAX_OUTPUT_TOKENS)

print(f"✓ Train: {len(train_ds)} samples  |  Val: {len(val_ds)} samples")


✓ Train: 165 samples  |  Val: 30 samples


## Step 8 — Fine-tune

In [12]:
# ── STEP 8 : TRAIN ───────────────────────────────────────────────────────────

training_args = Seq2SeqTrainingArguments(
    output_dir             = MODEL_OUTPUT_DIR,
    num_train_epochs       = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate          = LEARNING_RATE,
    warmup_steps           = max(10, len(train_ds) // (BATCH_SIZE * 4)),
    weight_decay           = 0.01,
    logging_dir            = os.path.join(MODEL_OUTPUT_DIR, "logs"),
    logging_steps          = 10,
    eval_strategy          = "epoch",
    save_strategy          = "epoch",
    load_best_model_at_end = True,
    predict_with_generate  = True,
    generation_max_length  = MAX_OUTPUT_TOKENS,
    fp16                   = torch.cuda.is_available(),  # True only when GPU available
    use_cpu = True,
    report_to              = "none",
    dataloader_num_workers = 0,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
)

trainer = Seq2SeqTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    # tokenizer     = tokenizer,
    data_collator = data_collator,
)

print(f"✓ Trainer configured")
print(f"  Epochs       : {NUM_EPOCHS}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  LR           : {LEARNING_RATE}")
print(f"  Train steps  : ~{len(train_ds) // BATCH_SIZE * NUM_EPOCHS}")
print()
print("⏱ Starting training  (CPU ≈ 1–5 min / epoch for 100 samples) ...")
trainer.train()
print("\n✓ Training complete!")


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✓ Trainer configured
  Epochs       : 3
  Batch size   : 2
  LR           : 0.0003
  Train steps  : ~246

⏱ Starting training  (CPU ≈ 1–5 min / epoch for 100 samples) ...


Epoch,Training Loss,Validation Loss
1,8.201959,7.450438
2,7.420044,6.692389
3,7.280110,6.522795



✓ Training complete!


In [13]:
# ── STEP 9 : SAVE BEST MODEL ─────────────────────────────────────────────────
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

# Also save the feature schema so downstream code can reconstruct null fields
schema_path = os.path.join(MODEL_OUTPUT_DIR, "clinical_schema.json")
with open(schema_path, 'w', encoding='utf-8') as f:
    json.dump(list(CLINICAL_SCHEMA.keys()), f, ensure_ascii=False, indent=2)

print(f"✓ Model saved  → {MODEL_OUTPUT_DIR}")
print(f"✓ Schema saved → {schema_path}")
print(f"  Files: {sorted(os.listdir(MODEL_OUTPUT_DIR))}")


✓ Model saved  → ./finetuned_model
✓ Schema saved → ./finetuned_model/clinical_schema.json
  Files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'checkpoint-166', 'checkpoint-249', 'checkpoint-83', 'clinical_schema.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


## Step 10 — Inference

`extract_features_from_label(label_path)` is the main entry point:
1. Loads the label file with `load_transcriptions`
2. Runs `postprocess_ocr` (exact pipeline)
3. Passes the cleaned text to the fine-tuned model
4. Merges the model output with the regex-extracted features
5. Returns a complete JSON and saves it to disk


In [14]:
# ── STEP 10 : INFERENCE FUNCTIONS ────────────────────────────────────────────

def load_finetuned_model(checkpoint_dir: str):
    """Load the saved fine-tuned model and tokenizer."""
    tok   = AutoTokenizer.from_pretrained(checkpoint_dir)
    mdl   = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_dir)
    mdl   = mdl.to(DEVICE)
    mdl.eval()
    print(f"✓ Loaded model from {checkpoint_dir}")
    return mdl, tok


def generate_json_from_text(
    cleaned_text: str,
    mdl,
    tok,
    max_new_tokens: int = 512,
) -> Dict:
    """
    Run the fine-tuned model on cleaned text and return a merged feature dict.

    Strategy:
    1. Regex-based extraction (fast, deterministic) fills obvious fields.
    2. Model generation fills remaining / harder fields.
    3. The two dicts are merged (model values override regex where both present).
    4. The complete schema is returned (null for any unfilled field).
    """
    # Step 1: regex baseline
    regex_features = extract_features(cleaned_text)

    # Step 2: model
    prompt = build_prompt(cleaned_text)
    inputs = tok(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT_TOKENS,
        truncation=True,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = mdl.generate(
            **inputs,
            max_new_tokens  = max_new_tokens,
            num_beams       = 4,
            early_stopping  = True,
            no_repeat_ngram_size = 3,
        )

    raw_output = tok.decode(out_ids[0], skip_special_tokens=True).strip()

    # Step 3: parse model JSON
    model_features = {}
    # Try full string first, then find first {...} block
    for candidate in [raw_output, re.search(r'\{.*\}', raw_output, re.DOTALL)]:
        if candidate is None:
            continue
        text_to_parse = candidate if isinstance(candidate, str) else candidate.group(0)
        try:
            model_features = json.loads(text_to_parse)
            break
        except json.JSONDecodeError:
            pass

    # Step 4: merge (model values win over regex when both present)
    merged = copy.deepcopy(CLINICAL_SCHEMA)
    for k, v in regex_features.items():
        if v is not None:
            merged[k] = v
    for k, v in model_features.items():
        if k in merged and v not in (None, "", "null", "non mentionné"):
            merged[k] = v

    return merged


def extract_features_from_label(
    label_path: str,
    mdl=None,
    tok=None,
    output_dir: str = "./extracted_features",
) -> Dict:
    """
    Full pipeline for a single label file:
    load → postprocess_ocr → model → save JSON.

    Args:
        label_path : path to a *_label.txt file
        mdl, tok   : fine-tuned model + tokenizer (pass None to use only regex)
        output_dir : folder where the JSON file is saved

    Returns:
        features dict
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1. Load & clean (exact postprocess_ocr pipeline)
    transcriptions = load_transcriptions(str(label_path), transcription_key='transcription')
    if transcriptions:
        cleaned = postprocess_ocr(
            transcriptions, abbreviations, medications, terms, tests, NOISE_PATTERNS
        )
    else:
        # Fallback: read raw text and use plain-text cleaner
        with open(label_path, 'r', encoding='utf-8') as f:
            raw = f.read()
        # Strip tag markers
        raw = re.sub(r'<\|ref\|>.*?<\|/ref\|>', '', raw, flags=re.DOTALL)
        raw = re.sub(r'<\|det\|>.*?<\|/det\|>', '', raw, flags=re.DOTALL)
        raw = re.sub(r'###', '', raw)
        cleaned = postprocess_plain_text(raw)

    print(f"  Cleaned text ({len(cleaned)} chars): {cleaned[:150]}...")

    # 2. Generate features
    if mdl is not None and tok is not None:
        features = generate_json_from_text(cleaned, mdl, tok)
    else:
        features = extract_features(cleaned)
        # Fill complete schema
        full = copy.deepcopy(CLINICAL_SCHEMA)
        full.update({k: v for k, v in features.items() if v is not None})
        features = full

    # 3. Save JSON
    out_name = Path(label_path).stem + "_features.json"
    out_path = os.path.join(output_dir, out_name)
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(features, f, ensure_ascii=False, indent=2)

    print(f"  ✓ Saved → {out_path}")
    return features


# ── Demo inference on all label files ────────────────────────────────────────
print("=== Demo Inference ===")
print("Loading saved model...")
inf_model, inf_tok = load_finetuned_model(MODEL_OUTPUT_DIR)

label_files = sorted(Path(DATA_ROOT).rglob("*_label.txt"))
demo_files  = label_files[:3]   # first 3 for display

if not demo_files:
    print("⚠ No label files found — running demo on synthetic text")
    # Synthetic demo
    _fake_label = "./demo_label.txt"
    with open(_fake_label, 'w') as f:
        f.write(
            "20/10/2016\tAge: 72 ans - Madame - ATCDF: mère DCD d'une démence\n"
            "MMSE: 18/30 - MoCA: 14 - FAQ: 12 - donépézil 5mg - HTA, diabète"
        )
    demo_files = [Path(_fake_label)]

for lf in demo_files:
    print(f"\n── {lf.name} ──────────────────────────────────────────────")
    feats = extract_features_from_label(str(lf), mdl=inf_model, tok=inf_tok)
    non_null = {k: v for k, v in feats.items() if v is not None}
    print("  Extracted fields:")
    print(json.dumps(non_null, ensure_ascii=False, indent=4))


=== Demo Inference ===
Loading saved model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights: 0it [00:00, ?it/s]

T5ForConditionalGeneration LOAD REPORT from: ./finetuned_model
Key                                                                                     | Status     | 
----------------------------------------------------------------------------------------+------------+-
base_model.model.encoder.block.{0...11}.layer.0.SelfAttention.v.lora_A.default.weight   | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.1.EncDecAttention.v.lora_B.default.weight | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.1.EncDecAttention.v.lora_A.default.weight | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.0.SelfAttention.v.lora_B.default.weight   | UNEXPECTED | 
base_model.model.encoder.block.{0...11}.layer.0.SelfAttention.q.lora_B.default.weight   | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.0.SelfAttention.q.lora_B.default.weight   | UNEXPECTED | 
base_model.model.decoder.block.{0...11}.layer.0.SelfAttention.q.lora_A.default.weight   | UNEXPECTED | 
b

✓ Loaded model from ./finetuned_model

── 2016_001_001_label.txt ──────────────────────────────────────────────
  Cleaned text (618 chars): Le 20/10/2016
- âge 72 ans
- accès hypertension artérielle évoluant depuis 04 ans mois
- de - troubles mnésique installation progressif depuis
04 mois...
  ✓ Saved → ./extracted_features/2016_001_001_label_features.json
  Extracted fields:
{
    "age": 72,
    "PTEDUCAT": 0,
    "PTWORK": "retraité",
    "VISDATE": "20/10/2016",
    "MH4CARD": 1,
    "KEYMED": "b, e",
    "CMMED": "b"
}

── 2016_001_002_label.txt ──────────────────────────────────────────────
  Cleaned text (726 chars): irm cérébrale atrophie hippocampique
+ méningite de la
Faux
. bilan biologique = Normal vitamine b 12 - b 9 tous
- lipidograme
- tdm zoloft comprimé 5...
  ✓ Saved → ./extracted_features/2016_001_002_label_features.json
  Extracted fields:
{
    "age": 69,
    "PTEDUCAT": 0,
    "VISDATE": "28/11/2016",
    "MH4CARD": 1,
    "KEYMED": "athymil, b, e, mag, vitamine, 

## Step 11 — Batch Process All Label Files

In [15]:
# ── STEP 11 : BATCH PROCESS ALL LABEL FILES ─────────────────────────────────

def process_all_label_files(
    data_root: str,
    mdl,
    tok,
    output_dir: str = "./extracted_features",
) -> List[Dict]:
    """
    Process every *_label.txt under data_root and save one JSON per file.
    Returns list of all feature dicts.
    """
    label_files = sorted(Path(data_root).rglob("*_label.txt"))
    print(f"Processing {len(label_files)} label files...")

    all_results = []
    for lf in label_files:
        print(f"  → {lf.name}")
        try:
            feats = extract_features_from_label(str(lf), mdl=mdl, tok=inf_tok,
                                                output_dir=output_dir)
            feats['_source_file'] = str(lf)
            all_results.append(feats)
        except Exception as e:
            print(f"  ⚠ Failed: {e}")

    # Save a combined summary
    summary_path = os.path.join(output_dir, "all_patients_features.json")
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print(f"\n✓ Processed {len(all_results)} files")
    print(f"✓ Combined summary → {summary_path}")
    return all_results


# Uncomment to run on the full dataset:
# all_features = process_all_label_files(DATA_ROOT, inf_model, inf_tok)
print("✓ Batch processing function ready.")
print("  Uncomment the last line to process all files.")


✓ Batch processing function ready.
  Uncomment the last line to process all files.


## Summary of Changes vs Previous Version

| Issue | Fix |
|---|---|
| Wrong cleaning | Now uses the **exact** `postprocess_ocr` from `OCR_postprocessing__4_.ipynb`, including `load_transcriptions`, spatial sort, line grouping, and index-tracked spell-check |
| Wrong label format | `load_transcriptions` now handles **both** poly_gt tab-JSON format and tag-delimited `<|ref|>…<|/det|>` format |
| Wrong JSON schema | Output is now the **complete 120+ field clinical schema** matching your variable list exactly |
| Weak feature extraction | 30+ regex/keyword rules cover dates, age, gender, MMSE, MoCA, FAQ, biomarkers, medications, family history, medical history flags |
| Master corpus misuse | Entries are cleaned through `postprocess_plain_text` and features extracted before training — only entries that yield ≥1 non-null field are kept |
| Model output format | Model is prompted to return a compact JSON with only non-null fields; inference merges model + regex outputs for robustness |
| No output file | `extract_features_from_label()` saves a `.json` file per document and an `all_patients_features.json` summary |
